In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

df = pd.read_csv("../../data/SpotGenTrack/Data Sources/spotify_tracks.csv", index_col=0)
print(df.shape)
print(df.head())
print(df.info())
print(df.describe())
import sys
from pathlib import Path

repo_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "src" / "training" / "data_utils.py").is_file()
)
sys.path.insert(0, str(repo_root / "src"))
from training.data_utils import grouped_track_id_split


In [ ]:
# Check missing values
print(df.isnull().sum())
df = df.dropna()  # or df.fillna(df.mean())

# Remove duplicates
df = df.drop_duplicates()
df = df.drop(columns=["track_number", "disc_number"])

# Identify your target variable (popularity)
target = 'popularity'  # Change to your column name

# Identify audio features (exclude non-numeric/ID columns)
audio_features = df.select_dtypes(include=[np.number]).columns.tolist()

if target in audio_features:
    audio_features.remove(target)
    
track_ids = df["id"].astype(str)
y = df[target]
df = df[audio_features]

print(f"Numerical features only: {len(audio_features)}")
print(f"Features: {audio_features}")
print(f"Target: {target}")

In [ ]:
# SPLIT FIRST using the shared artist/album-isolated 70/15/15 assignment
X = df
train_ids, validation_ids, test_ids = map(
    set, grouped_track_id_split(track_ids.tolist())
)
train_mask = track_ids.isin(train_ids)
validation_mask = track_ids.isin(validation_ids)
test_mask = track_ids.isin(test_ids)

X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_val, y_val = X.loc[validation_mask], y.loc[validation_mask]
X_test, y_test = X.loc[test_mask], y.loc[test_mask]

# NOW analyze correlations on TRAIN set only
train_df = X_train.copy()
train_df[target] = y_train

correlations = train_df[audio_features + [target]].corr(numeric_only=True)[target].sort_values(ascending=False)
correlations = correlations.drop(target)

print(correlations)

# Visualize
plt.figure(figsize=(10, 6))
correlations.head(15).plot(kind='barh')
plt.xlabel('Correlation with Popularity')
plt.title('Train Set - Feature Correlations')
plt.tight_layout()
plt.show()


In [ ]:
# ===== TRAIN RANDOM FOREST =====
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=20,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

# ===== EVALUATE =====

# Predictions
y_train_pred = rf.predict(X_train)
y_val_pred = rf.predict(X_val)
y_test_pred = rf.predict(X_test)

# Metrics
train_r2 = r2_score(y_train, y_train_pred)
val_r2 = r2_score(y_val, y_val_pred)
test_r2 = r2_score(y_test, y_test_pred)

train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

train_mae = mean_absolute_error(y_train, y_train_pred)
val_mae = mean_absolute_error(y_val, y_val_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

print(f"Train R2: {train_r2:.4f}")
print(f"Validation R2: {val_r2:.4f}")
print(f"Test R2: {test_r2:.4f}")

print(f"Train RMSE: {train_rmse:.4f}")
print(f"Validation RMSE: {val_rmse:.4f}")
print(f"Test RMSE: {test_rmse:.4f}")

print(f"Train MAE: {train_mae:.4f}")
print(f"Validation MAE: {val_mae:.4f}")
print(f"Test MAE: {test_mae:.4f}")

In [ ]:
importance = pd.Series(
    rf.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(importance)

In [ ]:
importance.plot.bar(figsize=(10,5))
plt.ylabel("Importance")
plt.title("Random Forest Feature Importance")
plt.tight_layout()
plt.show()